<div style="text-align: center;">
    <h1 style="color: #FF6347;">Self-Guided Lab: Retrieval-Augmented Generation (RAGs)</h1>
</div>

<div style="text-align: center;">
    <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExZ3FsdzRveTBrenMxM3VnbDMwaTJxN2NnZm50aGFibXk1NzNnY2Q0MCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/LR5ZBwZHv02lmpVoEU/giphy.gif" alt="NLP Gif" style="width: 300px; height: 150px; object-fit: cover; object-position: center;">
</div>

<h1 style="color: #FF6347;">Data Storage & Retrieval</h1>


<h2 style="color: #FF8C00;">PyPDFLoader</h2>

`PyPDFLoader` is a lightweight Python library designed to streamline the process of loading and parsing PDF documents for text processing tasks. It is particularly useful in Retrieval-Augmented Generation workflows where text extraction from PDFs is required.

- **What Does PyPDFLoader Do?**
  - Extracts text from PDF files, retaining formatting and layout.
  - Simplifies the preprocessing of document-based datasets.
  - Supports efficient and scalable loading of large PDF collections.

- **Key Features:**
  - Compatible with popular NLP libraries and frameworks.
  - Handles multi-page PDFs and embedded images (e.g., OCR-compatible setups).
  - Provides flexible configurations for structured text extraction.

- **Use Cases:**
  - Preparing PDF documents for retrieval-based systems in RAGs.
  - Automating the text extraction pipeline for document analysis.
  - Creating datasets from academic papers, technical manuals, and reports.


In [ ]:
%pip install langchain langchain_community pypdf
%pip install termcolor langchain_openai langchain-huggingface sentence-transformers chromadb langchain_chroma tiktoken openai python-dotenv


In [7]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

import warnings
warnings.filterwarnings('ignore')


<h3 style="color: #FF8C00;">Loading the Documents</h3>

In [8]:
# File path for the document

file_path = r"C:\Users\diogo\IRONHACK\Bootcamp\WEEK 7\DAY 5\LAB\lab-intro-rag\ai-for-everyone.pdf"

<h3 style="color: #FF8C00;">Documents into pages</h3>

The `PyPDFLoader` library allows efficient loading and splitting of PDF documents into smaller, manageable parts for NLP tasks.

This functionality is particularly useful in workflows requiring granular text processing, such as Retrieval-Augmented Generation (RAG).


In [9]:
# Load and split the document
loader = PyPDFLoader(file_path)
pages = loader.load_and_split()
len(pages)

297

<h3 style="color: #FF8C00;">Pages into Chunks</h3>


####  RecursiveCharacterTextSplitter in LangChain

The `RecursiveCharacterTextSplitter` is the **recommended splitter** in LangChain when you want to break down long documents into smaller, semantically meaningful chunks — especially useful in **RAG pipelines**, where clean context chunks lead to better LLM responses.

####  Parameters

| Parameter       | Description                                                                 |
|-----------------|-----------------------------------------------------------------------------|
| `chunk_size`    | The **maximum number of characters** allowed in a chunk (e.g., `1000`).     |
| `chunk_overlap` | The number of **overlapping characters** between consecutive chunks (e.g., `200`). This helps preserve context continuity. |

####  How it works
`RecursiveCharacterTextSplitter` attempts to split the text **intelligently**, trying the following separators in order:
1. Paragraphs (`"\n\n"`)
2. Lines (`"\n"`)
3. Sentences or words (`" "`)
4. Individual characters (as a last resort)

This makes it ideal for handling **natural language documents**, such as PDFs, articles, or long reports, without breaking sentences or paragraphs in awkward ways.



In [10]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(pages)

len(chunks)

1096

####  Alternative: CharacterTextSplitter

`CharacterTextSplitter` is a simpler splitter that breaks text into chunks based **purely on character count**, without trying to preserve any natural language structure.

##### Example:
```python
from langchain.text_splitter import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
````

This method is faster and more predictable but may split text in the middle of a sentence or paragraph, which can hurt performance in downstream tasks like retrieval or QA.

---

#### Comparison Table

| Feature                        | RecursiveCharacterTextSplitter | CharacterTextSplitter     |
| ------------------------------ | ------------------------------ | ------------------------- |
| Structure-aware splitting      |  Yes                          |  No                      |
| Preserves sentence/paragraphs  |  Yes                          |  No                      |
| Risk of splitting mid-sentence |  Minimal                     |  High                   |
| Ideal for RAG/document QA      |  Highly recommended           |  Only if structured text |
| Performance speed              |  Slightly slower             |  Faster                  |

---

#### Recommendation

Use `RecursiveCharacterTextSplitter` for most real-world document processing tasks, especially when building RAG pipelines or working with structured natural language content like PDFs or articles.

## Best Practices for Choosing Chunk Size in RAG

### Best Practices for Chunk Size in RAG

| Factor                      | Recommendation                                                                                                                                                                                          |
| ---------------------------| ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **LLM context limit**       | Choose a chunk size that lets you retrieve multiple chunks **without exceeding the model’s token limit**. For example, GPT-4o supports 128k tokens, but with GPT-3.5 (16k) or GPT-4 (32k), keep it modest. |
| **Chunk size (in characters)** | Typically: **500–1,000 characters** per chunk → ~75–200 tokens. This fits well for retrieval + prompt without context overflow.                                                                           |
| **Chunk size (in tokens)**  | If using token-based splitter (e.g. `TokenTextSplitter`): aim for **100–300 tokens** per chunk.                                                                                                            |
| **Chunk overlap**           | Use **overlap of 10–30%** (e.g., 100–300 characters or ~50 tokens) to preserve context across chunk boundaries and avoid cutting off important ideas mid-sentence.                                        |
| **Document structure**      | Use **`RecursiveCharacterTextSplitter`** to preserve semantic boundaries (paragraphs, sentences) instead of arbitrary cuts.                                                                                |
| **Task type**               | For **question answering**, smaller chunks (~500–800 chars) reduce noise.<br>For **summarization**, slightly larger chunks (~1000–1500) are OK.                                                          |
| **Embedding model**         | Some models (e.g., `text-embedding-3-large`) can handle long input. But still, smaller chunks give **finer-grained retrieval**, which improves relevance.                                                  |
| **Query type**              | If users ask **very specific questions**, small focused chunks are better. For broader queries, bigger chunks might help.                                                                                  |


### Rule of Thumb

| Use Case                 | Chunk Size      | Overlap |
| ------------------------| --------------- | ------- |
| Factual Q&A              | 500–800 chars   | 100–200 |
| Summarization            | 1000–1500 chars | 200–300 |
| Technical documents      | 400–700 chars   | 100–200 |
| Long reports/books       | 800–1200 chars  | 200–300 |
| Small LLMs (≤16k tokens) | ≤800 chars      | 100–200 |


### Avoid

- Chunks >2000 characters: risks context overflow.
- No overlap: may lose key information between chunks.



<h2 style="color: #FF8C00;">Embeddings</h2>

Embeddings transform text into dense vector representations, capturing semantic meaning and contextual relationships. They are essential for efficient document retrieval and similarity analysis.

- **What are OpenAI Embeddings?**
  - Pre-trained embeddings like `text-embedding-3-large` generate high-quality vector representations for text.
  - Encapsulate semantic relationships in the text, enabling robust NLP applications.

- **Key Features of `text-embedding-3-large`:**
  - Large-scale embedding model optimized for accuracy and versatility.
  - Handles diverse NLP tasks, including retrieval, classification, and clustering.
  - Ideal for applications with high-performance requirements.

- **Benefits:**
  - Reduces the need for extensive custom training.
  - Provides state-of-the-art performance in retrieval-augmented systems.
  - Compatible with RAGs to create powerful context-aware models.


In [12]:
from langchain_community.embeddings import OpenAIEmbeddings
from dotenv import load_dotenv

In [13]:
load_dotenv()

True

In [14]:
api_key = os.getenv("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

<h2 style="color: #FF8C00;">ChromaDB</h2>

ChromaDB is a versatile vector database designed for efficiently storing and retrieving embeddings. It integrates seamlessly with embedding models to enable high-performance similarity search and context-based retrieval.

### Workflow Overview:
- **Step 1:** Generate embeddings using a pre-trained model (e.g., OpenAI's `text-embedding-3-large`).
- **Step 2:** Store the embeddings in ChromaDB for efficient retrieval and similarity calculations.
- **Step 3:** Use the stored embeddings to perform searches, matching, or context-based retrieval.

### Key Features of ChromaDB:
- **Scalability:** Handles large-scale datasets with optimized indexing and search capabilities.
- **Speed:** Provides fast and accurate retrieval of embeddings for real-time applications.
- **Integration:** Supports integration with popular frameworks and libraries for embedding generation.

In [16]:
from langchain_community.vectorstores import Chroma

In [17]:
db = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db_LAB")
print("ChromaDB created with document embeddings.")

ChromaDB created with document embeddings.


<h1 style="color: #FF6347;">Retrieving Documents</h1>


### Exercice1: Write a user question that someone might ask about your book’s topic or content.

In [44]:
user_question = "Will AI replace all human work?" # User question
retrieved_docs = db.similarity_search(user_question, k=10) # k is the number of documents to retrieve

In [45]:
# Display top results
for i, doc in enumerate(retrieved_docs[:3]): # Display top 3 results
    print(f"Document {i+1}:\n{doc.page_content[36:1000]}") # Display content

Document 1:
will need to coexist with machines. Jobs traditionally done by 
humans will be shifted towards AI systems. Artificial intelligence is already able 
to translate languages, diagnose illnesses, assist in retail (Kaplan 2020c), and  
the like – in several cases, better than the human workforce. Human jobs might 
be created in the future that are unimaginable now, similar to the fact that 
nobody really predicted the job of mobile app designers just a few years ago.
In this world, AI would rather be augmenting and complementing – rather 
than replacing – humans in their work. In the pessimistic case, i.e., massive 
unemployment, ideas such as universal basic income are already being dis -
cussed. Fundamental philosophical questions would need to be answered sur-
rounding life for humans when most of our work is done by AI systems. In 
any case, the State will certainly have to come up with a set of rules govern -
Document 2:
oyees with AI, as this would not be a long-
term stra

<h2 style="color: #FF8C00;">Preparing Content for GenAI</h2>

In [46]:
def _get_document_prompt(docs):
    prompt = "\n"
    for doc in docs:
        prompt += "\nContent:\n"
        prompt += doc.page_content + "\n\n"
    return prompt

In [47]:
# Generate a formatted context from the retrieved documents
formatted_context = _get_document_prompt(retrieved_docs)
print("Context formatted for GPT model.")

Context formatted for GPT model.


<h2 style="color: #FF8C00;">ChatBot Architecture</h2>

### Exercice2: Write a prompt that is relevant and tailored to the content and style of your book.

In [48]:
prompt = f"""
## SYSTEM ROLE
You are a precise and factual assistant that explains **Artificial Intelligence for Everyone** in a simple way.
You MUST answer **only** using the information in the CONTEXT section.

If the answer is not fully supported by the CONTEXT, you MUST reply exactly:
"The provided context does not contain this information."

## USER QUESTION
"{user_question}"

## CONTEXT
Use ONLY the following content to answer:
'''
{formatted_context}
'''

## STRICT RULES
1. **Use ONLY Context**
   - Do NOT use outside knowledge.
   - Do NOT guess, assume, or infer anything that is not clearly stated in the context.
   - If the context is insufficient, respond with:
     "The provided context does not contain this information."

2. **Answer Structure**
   - Start with a short, plain-language explanation of the main concept.
   - Then use bullet points to break it down, following this structure **only if the context allows**:
     - **Definition**
     - **Purpose / Why it matters**
     - **How it works (high level)**
     - **Examples or use cases**
     - **Benefits and limitations**
   - If present in the context, add:
     - Practical implications for everyday people or businesses.
     - Common misunderstandings and clarifications.

3. **Transparency and Sources**
   - When you use a piece of information, reference the source title and page(s) if they are provided in the context.
   - Never invent a source, title, or page number.

4. **Clarity**
   - Use simple, friendly, beginner-level language.
   - Avoid equations, code, or jargon unless clearly explained in the context.
   - Format the answer in Markdown.

## TASK
Provide the final answer in exactly this format:

'''
# [Short, Clear Title]
[Answer in simple language, based ONLY on the context.]

**Source:**
• [Source Title], Pages [...]
'''
"""
print("Prompt constructed.")


Prompt constructed.


In [49]:
import openai

### Exercice3: Tune parameters like temperature, and penalties to control how creative, focused, or varied the model's responses are.

In [63]:
# Set up GPT client and parameters
model_params = {
    'model': 'gpt-4o',
    'temperature': 0.5,       # No creativity
    'max_output_tokens': 600,
    'top_p': 1.0,
    'frequency_penalty': 0.0,
    'presence_penalty': 0.0
}


<h1 style="color: #FF6347;">Response</h1>


In [64]:
from openai import OpenAI
client = OpenAI()

messages = [
    {"role": "user", "content": prompt}
]

response = client.responses.create(
    input=messages,
    model=model_params['model'],
    temperature=model_params['temperature'],
    max_output_tokens=model_params['max_output_tokens'], 
    top_p=model_params['top_p'],
    timeout=120
)

print(response.output_text)

'''
# Will AI Replace All Human Work?
The context suggests that AI will not replace all human work but will likely augment and complement it. AI might take over some jobs, but new types of jobs could also be created.

- **AI and Human Coexistence**: Humans will need to coexist with machines, with AI systems taking over some tasks traditionally done by humans.
- **Job Shifts**: While AI can perform certain tasks better than humans, it is expected to augment human work rather than completely replace it.
- **Future Job Creation**: New jobs might emerge that are currently unimaginable, similar to how mobile app design jobs emerged.
- **Collaborative Intelligence**: The best outcomes are seen when humans and AI work together, enhancing each other's strengths.
- **Regulation and Adaptation**: Regulations could help manage job transitions, and ideas like universal basic income are being discussed if massive unemployment occurs.

**Source:**
• AI for Everyone?, Pages 22, 28, 218, 252
'''


In [65]:
print(response.output[0].content[0].text)


'''
# Will AI Replace All Human Work?
The context suggests that AI will not replace all human work but will likely augment and complement it. AI might take over some jobs, but new types of jobs could also be created.

- **AI and Human Coexistence**: Humans will need to coexist with machines, with AI systems taking over some tasks traditionally done by humans.
- **Job Shifts**: While AI can perform certain tasks better than humans, it is expected to augment human work rather than completely replace it.
- **Future Job Creation**: New jobs might emerge that are currently unimaginable, similar to how mobile app design jobs emerged.
- **Collaborative Intelligence**: The best outcomes are seen when humans and AI work together, enhancing each other's strengths.
- **Regulation and Adaptation**: Regulations could help manage job transitions, and ideas like universal basic income are being discussed if massive unemployment occurs.

**Source:**
• AI for Everyone?, Pages 22, 28, 218, 252
'''


<img src="https://miro.medium.com/v2/resize:fit:824/1*GK56xmDIWtNQAD_jnBIt2g.png" alt="NLP Gif" style="width: 500px">

<h2 style="color: #FF6347;">Cosine Similarity</h2>

**Cosine similarity** is a metric used to measure the alignment or similarity between two vectors, calculated as the cosine of the angle between them. It is the **most common metric used in RAG pipelines** for vector retrieval.. It provides a scale from -1 to 1:

- **-1**: Vectors are completely opposite.
- **0**: Vectors are orthogonal (uncorrelated or unrelated).
- **1**: Vectors are identical.


<img src="https://storage.googleapis.com/lds-media/images/cosine-similarity-vectors.original.jpg" alt="NLP Gif" style="width: 700px">

<h2 style="color: #FF6347;">Keyword Highlighting</h2>

Highlighting important keywords helps users quickly understand the relevance of the retrieved text to their query.

In [62]:
from termcolor import colored

The `highlight_keywords` function is designed to highlight specific keywords within a given text. It replaces each keyword in the text with a highlighted version using the `colored` function from the `termcolor` library.


In [66]:
def highlight_keywords(text, keywords):
    for keyword in keywords:
        text = text.replace(keyword, colored(keyword, 'green', attrs=['bold']))
    return text

### Exercice4: add your keywords

In [71]:
query_keywords = ["job","human","replace"] # add your keywords
for i, doc in enumerate(retrieved_docs[:1]):
    snippet = doc.page_content[:1000]
    highlighted = highlight_keywords(snippet, query_keywords)
    print(f"Snippet {i+1}:\n{highlighted}\n{'-'*80}")

Snippet 1:
22 AI for Everyone?
Clearly, humans will need to coexist with machines. Jobs traditionally done by 
humans will be shifted towards AI systems. Artificial intelligence is already able 
to translate languages, diagnose illnesses, assist in retail (Kaplan 2020c), and  
the like – in several cases, better than the human workforce. Human jobs might 
be created in the future that are unimaginable now, similar to the fact that 
nobody really predicted the job of mobile app designers just a few years ago.
In this world, AI would rather be augmenting and complementing – rather 
than replacing – humans in their work. In the pessimistic case, i.e., massive 
unemployment, ideas such as universal basic income are already being dis -
cussed. Fundamental philosophical questions would need to be answered sur-
rounding life for humans when most of our work is done by AI systems. In 
any case, the State will certainly have to come up with a set of rules govern -
------------------------------

1. `query_keywords` is a list of keywords to be highlighted.
2. The loop iterates over the first document in retrieved_docs.
3. For each document, a snippet of the first 200 characters is extracted.
4. The highlight_keywords function is called to highlight the keywords in the snippet.
5. The highlighted snippet is printed along with a separator line.

<h1 style="color: #FF6347;">Bonus</h1>

**Try loading one of your own PDF books and go through the steps again to explore how the pipeline works with your content**:
